# 16 — Temporal Forecast Validation

**LeukemiPrediag v2.3**

This notebook tests whether the longitudinal VAF engine can predict the **next observed VAF** from earlier samples, without using leukemia outcome labels.

It compares:

- **carry-forward:** assume no growth after the last training sample;
- **raw linear:** extrapolate VAF linearly;
- **log-linear:** exponential-growth approximation;
- **cross-fitted shrunk log-linear:** estimate growth from earlier samples but shrink it toward zero. The shrinkage weight is chosen only on training patients in each outer fold.

This is a measurement/prediction-validity test, **not a leukemia-risk validation**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MYDRIVE = Path("/content/drive/MyDrive")
CANDIDATES = [
    MYDRIVE / "leukemia-prediagnose" / "blood-cancer-early-detection",
    MYDRIVE / "blood-cancer-early-detection",
]
CANDIDATES += list(MYDRIVE.glob("*/blood-cancer-early-detection"))
PROJECT_ROOT = next((p for p in CANDIDATES if p.exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find blood-cancer-early-detection under MyDrive")

os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from temporal_validation import (
    leave_last_timepoint_forecasts, summarize_forecasts,
    cross_fitted_shrinkage_forecasts, summarize_cross_fitted_shrinkage,
    TemporalValidationConfig,
)

SERIAL_PATH = PROJECT_ROOT / "data" / "processed" / "v2" / "real_serial_filtered.csv"
OUTDIR = PROJECT_ROOT / "data" / "processed" / "v2" / "temporal_validation"
FIGDIR = PROJECT_ROOT / "results" / "temporal_validation_v23"
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("serial exists:", SERIAL_PATH.exists())
print("OUTDIR:", OUTDIR)

In [ ]:
serial = pd.read_csv(SERIAL_PATH)
print("rows:", len(serial))
print("patients:", serial.patient_id.nunique())
print("trajectories:", serial.groupby(["patient_id", "variant_id"]).ngroups)
print(serial.groupby("cohort").patient_id.nunique())

## A. Naive one-step-ahead validation

For every trajectory with at least three measurements, hold out the last measurement and fit only the earlier measurements.

In [ ]:
cfg = TemporalValidationConfig()
forecasts = leave_last_timepoint_forecasts(serial, cfg)
overall, by_cohort = summarize_forecasts(forecasts)

display(overall)
display(by_cohort)

forecasts.to_csv(OUTDIR / "leave_last_forecasts.csv", index=False)
overall.to_csv(OUTDIR / "naive_forecast_summary.csv")
by_cohort.to_csv(OUTDIR / "naive_forecast_by_cohort.csv", index=False)

In [ ]:
# Visual comparison of absolute log-VAF forecast error
cols = [
    "abs_log_error_carry_forward",
    "abs_log_error_raw_linear",
    "abs_log_error_log_linear",
]
plot_df = forecasts[cols].rename(columns={
    "abs_log_error_carry_forward": "Carry-forward",
    "abs_log_error_raw_linear": "Raw linear",
    "abs_log_error_log_linear": "Log-linear",
})

fig, ax = plt.subplots(figsize=(7,5))
ax.boxplot([plot_df[c].dropna() for c in plot_df.columns], labels=plot_df.columns, showfliers=False)
ax.set_ylabel("Absolute log-VAF forecast error")
ax.set_title("Naive one-step-ahead forecast error")
fig.tight_layout()
fig.savefig(FIGDIR / "01_naive_forecast_error.png", dpi=220, bbox_inches="tight")
plt.show()

## B. Cross-fitted shrinkage

A raw two-point slope is noisy. The model therefore tests a conservative version:



\(\log \hat VAF_{t+\Delta}=\log VAF_t + \alpha\,\hat s\,\Delta\)

where \(\alpha=0\) is carry-forward and \(\alpha=1\) is full log-linear extrapolation.

The shrinkage weight is tuned on **training patients only** inside each outer fold.

In [ ]:
crossfit, folds = cross_fitted_shrinkage_forecasts(
    serial, cfg, n_splits=5, seed=17
)
shrink_summary = summarize_cross_fitted_shrinkage(crossfit)

display(folds)
display(shrink_summary)

cohort_shrink = (
    crossfit.groupby("cohort")[[
        "abs_log_error_carry_forward",
        "abs_log_error_log_linear",
        "abs_log_error_shrunk_log_linear",
        "abs_error_carry_forward",
        "abs_error_shrunk_log_linear",
    ]].mean()
)
display(cohort_shrink)

crossfit.to_csv(OUTDIR / "cross_fitted_shrinkage_forecasts.csv", index=False)
folds.to_csv(OUTDIR / "cross_fitted_shrinkage_folds.csv", index=False)
shrink_summary.to_csv(OUTDIR / "cross_fitted_shrinkage_summary.csv")
cohort_shrink.to_csv(OUTDIR / "cross_fitted_shrinkage_by_cohort.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
vals = [
    crossfit["abs_log_error_carry_forward"].dropna(),
    crossfit["abs_log_error_log_linear"].dropna(),
    crossfit["abs_log_error_shrunk_log_linear"].dropna(),
]
ax.boxplot(vals, labels=["Carry-forward", "Log-linear", "Shrunk log-linear"], showfliers=False)
ax.set_ylabel("Absolute log-VAF forecast error")
ax.set_title("Patient-held-out temporal validation")
fig.tight_layout()
fig.savefig(FIGDIR / "02_crossfitted_shrinkage_error.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# Observed vs forecast VAF on log axes
fig, ax = plt.subplots(figsize=(6.5,6))
ax.scatter(crossfit["true_vaf"], crossfit["pred_shrunk_log_linear"], s=18, alpha=0.55)
lo = max(min(crossfit["true_vaf"].min(), crossfit["pred_shrunk_log_linear"].min()), 1e-4)
hi = max(crossfit["true_vaf"].max(), crossfit["pred_shrunk_log_linear"].max())
ax.plot([lo,hi],[lo,hi], linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Observed held-out VAF")
ax.set_ylabel("Shrunk forecast VAF")
ax.set_title("Held-out next-sample VAF forecast")
fig.tight_layout()
fig.savefig(FIGDIR / "03_observed_vs_shrunk_forecast.png", dpi=220, bbox_inches="tight")
plt.show()

## C. Interpretation

A model is not considered improved just because its fitted slope looks biologically plausible. A useful dynamics engine should also forecast future measurements.

The key comparison is the cross-fitted shrunk-growth model versus the carry-forward baseline. Any gain should be reported together with cohort heterogeneity. If a cohort does not improve, do not hide it.

This notebook validates **VAF dynamics only**. It does not establish prediction of AML/MDS or any other clinical outcome.

In [ ]:
audit = {
    "version": "LeukemiPrediag-v2.3",
    "research_use_only": True,
    "not_a_clinical_outcome_validation": True,
    "n_patients": int(serial.patient_id.nunique()),
    "n_eligible_trajectories": int(len(crossfit)),
    "fold_shrinkage_alpha": [float(x) for x in folds["shrinkage_alpha"]],
    "summary": shrink_summary.reset_index().to_dict(orient="records"),
    "interpretation": (
        "Forward VAF prediction is an outcome-free measurement-validity test. "
        "Clinical progression performance still requires an outcome-linked cohort."
    ),
}
with open(OUTDIR / "temporal_validation_audit.json", "w") as f:
    json.dump(audit, f, indent=2)
print(json.dumps(audit, indent=2))